# Data Preparation – Swiss weather, 2021–2025

## 1. Set up the study period and paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

raw_dir = root / "data" / "raw data"
raw_csv = raw_dir / "weather-data_2021-2025.csv"
SWISS_ZONE = "Europe/Zurich"
WEATHER_COLUMNS = ["temperature_2m", "shortwave_radiation"]
expected = pd.date_range("2021-01-01", "2026-01-01", freq="h",
                         tz=SWISS_ZONE, inclusive="left")

expected_sites = {"Zurich", "Basel", "Bern", "Geneva", "Sion", "Lugano", "Chur"}

## 2. Load the raw weather CSV
Use the site-level CSV exported by 01_data_understanding_weather. 

Each timestamp includes its winter or summer offset.

In [2]:
df = pd.read_csv(raw_csv, usecols=["timestamp", *WEATHER_COLUMNS, "Location"])
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert(SWISS_ZONE)
display(df.head(7))

,timestamp,temperature_2m,shortwave_radiation,Location
0,2021-01-01 00:00:00+01:00,0.7,0.0,Zurich
1,2021-01-01 00:00:00+01:00,1.8,0.0,Basel
2,2021-01-01 00:00:00+01:00,-1.5,0.0,Bern
3,2021-01-01 00:00:00+01:00,1.1,0.0,Geneva
4,2021-01-01 00:00:00+01:00,-1.9,0.0,Sion
5,2021-01-01 00:00:00+01:00,5.2,0.0,Lugano
6,2021-01-01 00:00:00+01:00,-0.9,0.0,Chur


## 3. Check Swiss local time
Read the explicit CSV offsets as absolute instants and display them in Europe/Zurich. 

The example shows both occurrences of 02:00 on the autumn clock-change day.

In [3]:
autumn = df.loc[df["Location"].eq("Zurich")].set_index("timestamp").sort_index()
display(autumn.loc["2024-10-27 00:00":"2024-10-27 04:00"])

,temperature_2m,shortwave_radiation,Location
timestamp,,,
2024-10-27 00:00:00+02:00,8.4,0.0,Zurich
2024-10-27 01:00:00+02:00,8.0,0.0,Zurich
2024-10-27 02:00:00+02:00,9.3,0.0,Zurich
2024-10-27 02:00:00+01:00,9.1,0.0,Zurich
2024-10-27 03:00:00+01:00,9.0,0.0,Zurich
2024-10-27 04:00:00+01:00,9.2,0.0,Zurich


## 4. Verify the five-year study period
Display the local time boundaries and check the record count for each year.

In [4]:
print(f"Local study period: {df['timestamp'].min()} to {df['timestamp'].max()}")
annual = df.groupby(df["timestamp"].dt.year).size().rename("Site-hours").to_frame()
annual["Expected site-hours"] = pd.Series(expected.year).value_counts() * len(expected_sites)
display(annual)

Local study period: 2021-01-01 00:00:00+01:00 to 2025-12-31 23:00:00+01:00


,Site-hours,Expected site-hours
timestamp,,
2021,61320,61320
2022,61320,61320
2023,61320,61320
2024,61488,61488
2025,61320,61320


## 5. Confirm seven observations per hour
Check the exact hourly calendar separately for every site. 

Each expected hour must have one finite value per variable at all seven locations.

In [5]:
if set(df["Location"].dropna()) != set(expected_sites) or df["Location"].isna().any():
    raise ValueError("Check the site names in the raw weather CSV.")
if not np.isfinite(df[WEATHER_COLUMNS].to_numpy(dtype=float)).all():
    raise ValueError("A weather measurement is missing or non-finite.")
for name, rows in df.groupby("Location"):
    hours = pd.DatetimeIndex(rows["timestamp"]).sort_values()
    if not hours.equals(expected):
        raise ValueError(f"{name}: check for missing, duplicate or unexpected hours.")
display(df.groupby("Location").size().rename("Hours").to_frame())

,Hours
Location,
Basel,43824
Bern,43824
Chur,43824
Geneva,43824
Lugano,43824
Sion,43824
Zurich,43824


## 6. Average the seven sites
Sum each weather variable across the seven sites for each timestamp, divide by seven, then round the result to four decimal places. 

Every location has equal weight.

In [6]:
hourly_mean = (df.groupby("timestamp", sort=True)[WEATHER_COLUMNS]
               .sum()
               .div(len(expected_sites))
               .round(4))
df_avg = hourly_mean.reset_index()
display(df_avg.head())

,timestamp,temperature_2m,shortwave_radiation
0,2021-01-01 00:00:00+01:00,0.6429,0.0
1,2021-01-01 01:00:00+01:00,0.8714,0.0
2,2021-01-01 02:00:00+01:00,0.4286,0.0
3,2021-01-01 03:00:00+01:00,0.3571,0.0
4,2021-01-01 04:00:00+01:00,0.1429,0.0


## 7. Check hourly coverage and measurements
The five-year calendar includes the 2024 leap day and all Swiss clock changes.

In [7]:
actual = pd.DatetimeIndex(df_avg['timestamp'])

for column in ['temperature_2m', 'shortwave_radiation']:
    df_avg[column] = pd.to_numeric(df_avg[column], errors='raise')

checks = pd.Series({
    'Missing expected hours': len(expected.difference(actual)),
    'Unexpected timestamps': len(actual.difference(expected)),
    'Duplicate timestamps': int(actual.duplicated().sum()),
    'Missing temperature values': int(df_avg['temperature_2m'].isna().sum()),
    'Missing radiation values': int(df_avg['shortwave_radiation'].isna().sum()),
})
display(checks.to_frame('Count'))
if checks.any():
    raise ValueError('Review the non-zero counts before exporting.')

,Count
Missing expected hours,0
Unexpected timestamps,0
Duplicate timestamps,0
Missing temperature values,0
Missing radiation values,0


## 8. Export the prepared weather data
Save the three columns using timestamp strings with seconds and explicit Swiss winter or summer offsets.

In [8]:
output_file = root / "data" / "processed data" / "weather-data_2021-2025.csv"
output_file.parent.mkdir(parents=True, exist_ok=True)
export = df_avg.copy()
export["timestamp"] = export["timestamp"].map(lambda t: t.isoformat(timespec="seconds"))
export.to_csv(output_file, index=False)
print(f"Saved {len(export):,} hourly records to {output_file.name}")

Saved 43,824 hourly records to weather-data_2021-2025.csv
